In [1]:
!git clone https://github.com/Ramyghr/MarketMind.git
%cd MarketMind
!pip install wandb -q
!pip install -r requirements.txt -q

fatal: destination path 'MarketMind' already exists and is not an empty directory.
/kaggle/working/MarketMind
ERROR: Could not install packages due to an OSError: [Errno 2] No such file or directory: '/home/task_176496125431831/conda-bld/brotlicffi_1764961387175/work'



In [2]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

True
Tesla T4


In [6]:
import os, shutil
os.makedirs('data/raw', exist_ok=True)
shutil.copy('/kaggle/input/datasets/tretn3/marketmind-data/BTCUSDT_1h.parquet', 'data/raw/BTCUSDT_1h.parquet')
shutil.copy('/kaggle/input/datasets/tretn3/marketmind-data/ETHUSDT_1h.parquet', 'data/raw/ETHUSDT_1h.parquet')
shutil.copy('/kaggle/input/datasets/tretn3/marketmind-data/SPY_1d.parquet', 'data/raw/SPY_1d.parquet')
shutil.copy('/kaggle/input/datasets/tretn3/marketmind-data/QQQ_1d.parquet', 'data/raw/QQQ_1d.parquet')
print('done')

done


In [8]:
import wandb
wandb.login(key='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')

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: ramy-gharbi (ramy-gharbi-esprit) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [9]:
import sys
sys.path.insert(0, 'src')

import torch
import wandb
from torch.utils.data import DataLoader
from ssl_model.encoder import MarketEncoder
from ssl_model.losses import nt_xent_loss
from ssl_model.augmentations import augment
from data.dataset import load_split

config = {
    'lr': 3e-4,
    'batch_size': 256,
    'epochs': 30,
    'temperature': 0.07,
    'embed_dim': 128,
    'data': '2018-2024'
}

wandb.init(project='marketmind', name='ssl-multi-asset-30ep', config=config)

paths = [
    'data/raw/BTCUSDT_1h.parquet',
    'data/raw/ETHUSDT_1h.parquet',
    'data/raw/SPY_1d.parquet',
    'data/raw/QQQ_1d.parquet',
]

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)

train_ds = load_split(paths, split='train', augment_fn=augment)
train_loader = DataLoader(train_ds, batch_size=config['batch_size'], shuffle=True, num_workers=0)

model = MarketEncoder().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=config['lr'])
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=config['epochs'])

import os
os.makedirs('checkpoints', exist_ok=True)

for epoch in range(config['epochs']):
    model.train()
    total_loss = 0
    for x1, x2 in train_loader:
        x1, x2 = x1.float().to(device), x2.float().to(device)
        z1, z2 = model(x1), model(x2)
        loss = nt_xent_loss(z1, z2, temperature=config['temperature'])
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        total_loss += loss.item()
        wandb.log({'batch_loss': loss.item()})
    epoch_loss = total_loss / len(train_loader)
    scheduler.step()
    wandb.log({'epoch_loss': epoch_loss, 'epoch': epoch})
    print(f'epoch {epoch+1} loss: {epoch_loss:.4f}')

torch.save({
    'epoch': config['epochs'],
    'model_state': model.state_dict(),
    'config': config,
    'final_loss': epoch_loss
}, 'checkpoints/ssl_encoder_30ep.pt')
print('checkpoint saved')

device: cuda
epoch 1 loss: 2.3111
epoch 2 loss: 0.9040
epoch 3 loss: 0.6484
epoch 4 loss: 0.5266
epoch 5 loss: 0.4612
epoch 6 loss: 0.4134
epoch 7 loss: 0.3747
epoch 8 loss: 0.3497
epoch 9 loss: 0.3300
epoch 10 loss: 0.3121
epoch 11 loss: 0.2986
epoch 12 loss: 0.2947
epoch 13 loss: 0.2752
epoch 14 loss: 0.2687
epoch 15 loss: 0.2603
epoch 16 loss: 0.2526
epoch 17 loss: 0.2478
epoch 18 loss: 0.2421
epoch 19 loss: 0.2381
epoch 20 loss: 0.2345
epoch 21 loss: 0.2309
epoch 22 loss: 0.2299
epoch 23 loss: 0.2273
epoch 24 loss: 0.2266
epoch 25 loss: 0.2239
epoch 26 loss: 0.2188
epoch 27 loss: 0.2218
epoch 28 loss: 0.2187
epoch 29 loss: 0.2199
epoch 30 loss: 0.2177
checkpoint saved


In [11]:
shutil.copy('checkpoints/ssl_encoder_30ep.pt', '/kaggle/working/ssl_encoder_30ep.pt')

'/kaggle/working/ssl_encoder_30ep.pt'